# **7일차 실습: 도메인 특화 도구 만들기 및 테스트**

## 학습 목표
1. LangChain Tool 개념 이해
2. AI를 활용한 도구 자동 생성
3. 생성한 도구 테스트
4. 도메인 특화 에이전트에 적용

## 실습 단계
1. **도구 설계**: 팀 프로젝트에 필요한 도구 정의
2. **도구 생성**: AI를 활용하여 도구 코드 자동 생성
3. **도구 테스트**: 생성된 도구가 올바르게 동작하는지 확인
4. **에이전트 적용**: domain-agent에 통합 (다음 단계)

---

## 1. AI 도구 생성 프롬프트 확인

팀 프로젝트에 필요한 도구를 AI로 자동 생성하기 위한 프롬프트입니다.

**프롬프트 파일 위치**: `../../make_tool_prompt.txt`

In [1]:
import os

prompt_path = "../make_tool_prompt.txt" if os.path.exists("../make_tool_prompt.txt") else "make_tool_prompt.txt"

with open(prompt_path, "r", encoding="utf-8") as f:
    prompt_template = f.read()

print("=" * 80)
print("AI 도구 생성 프롬프트")
print("=" * 80)
print(prompt_template)
print("\n✓ 프롬프트 로드 완료")
print("\n💡 이 프롬프트를 ChatGPT, Claude 등에 복사하여 사용하세요!")

AI 도구 생성 프롬프트
당신은 LangChain/LangGraph 기반 AI Agent를 위한 Python Tool 개발 전문가입니다.

아래 **[사용자 요구사항]** 에 작성된 내용을 바탕으로 LangChain Tool을 생성하세요.

---

# 사용자 요구사항

## 여기에 원하는 기능을 작성하세요.

<이곳에 원하는 Tool 기능을 작성>

---

# 구현 규칙

다음 규칙을 반드시 지키세요.

## 출력 형식

* Python 코드만 출력합니다.
* 코드 외의 설명은 출력하지 않습니다.
* `from langchain_core.tools import tool`을 사용합니다.
* `@tool(parse_docstring=True)` 데코레이터를 사용합니다.
* Python 3.11 이상 기준으로 작성합니다.

## 함수 작성 규칙

* 함수명은 기능에 맞게 작성합니다.
* 모든 매개변수에는 타입 힌트를 작성합니다.
* 반환 타입도 작성합니다.
* 필요한 import는 함수 내부에서 수행합니다.
* 가능한 표준 라이브러리를 우선 사용합니다.
* 외부 라이브러리가 필요한 경우 import도 함께 작성합니다.

## Docstring

반드시 Google Style Docstring을 작성합니다.

예시 형식

```python
"""도구 설명

Args:
    parameter1: 설명
    parameter2: 설명

Returns:
    반환값 설명
"""
```

## 예외 처리

반드시 예외 처리를 구현합니다.

형식

```python
try:
    ...
    return "성공 메시지"
except Exception as e:
    return f"실패: {str(e)}"
```

## 테스트 코드

마지막에 아래 코드를 추가합니다.

```python
print(f"도구 이름: {함수명.name}")
print(f"도구 설명: {함수명.description}")
```

## 코드 품질

* 읽기 쉬운 코드로 작성합니다.
* 

## 2. 도구 설계 가이드

### 좋은 도구의 조건

1. **단일 책임**: 하나의 명확한 기능만 수행
2. **명확한 입력/출력**: 매개변수와 반환값이 명확
3. **에러 처리**: 예외 상황을 적절히 처리
4. **좋은 설명**: Docstring으로 도구의 기능을 명확히 설명

### 도메인별 도구 예시

**쇼핑 도메인:**
- 상품 검색
- 가격 비교
- 재고 확인
- 리뷰 조회

**법령 도메인:**
- 법령 검색
- 조문 조회
- 판례 검색
- 법령 해석

**의료 도메인:**
- 증상 검색
- 병원 찾기
- 약 정보 조회
- 건강 정보 제공

**여행 도메인:**
- 항공권 검색
- 호텔 검색
- 관광지 정보
- 날씨 확인

---

## 3. 도구 생성 프로세스

### Step 1: 팀 프로젝트 도메인 및 필요한 도구 정의

**TODO: 팀에서 선택한 도메인과 필요한 도구를 작성하세요**

팀 도메인: 스마트 요리 추천 및 배달/레시피 에이전트

필요한 도구 목록:
1. 도구명: search_recipe_from_10000recipe (만개의레시피 레시피 검색)
   - 입력: menu_name (만들고자 하는 메뉴 이름)
   - 출력: 만개의레시피 검색 결과, 필수 재료 및 단계별 조리 순서 요약 (문자열)
   - 역할: '만개의레시피' 사이트에서 특정 메뉴의 상세 조리법과 재료 정보를 수집하여 텍스트로 제공

2. 도구명: search_nearby_restaurants (근처 맛집 추천 검색)
   - 입력: address (사용자의 현재 주소 또는 동 이름), menu_name (먹고 싶은 메뉴 이름)
   - 출력: 근처 맛집 목록 (4~5곳 이상), 간략한 소개, 평점 및 평가 요약 (문자열)
   - 역할: 사용자의 위치와 메뉴를 바탕으로 웹 검색을 수행하여 주변 맛집 정보와 후기를 수집하고 요약

### Step 2: AI로 도구 생성하기

**사용 방법:**

1. 위의 프롬프트 템플릿을 복사
2. `<이곳에 원하는 Tool 기능을 작성>` 부분에 팀의 도구 요구사항 작성
3. ChatGPT, Claude 등에 입력하여 코드 생성
4. 생성된 코드를 아래 셀에 붙여넣기

**예시 입력:**
```
쇼핑 도메인의 상품 검색 도구를 만들어주세요.

기능:
- 상품명으로 검색
- 가격 범위 필터링
- 카테고리 필터링
- 검색 결과를 JSON 형태로 반환
```

---

## 4. 생성된 도구 코드 테스트

**TODO: AI가 생성한 도구 코드를 아래에 붙여넣으세요**

**중요:** 
- 코드를 실행하기 전에 반드시 검토하세요
- 필요한 외부 라이브러리가 있다면 먼저 설치하세요
- 실제 API 키가 필요한 경우 .env 파일에 추가하세요

In [2]:
from langchain_core.tools import tool
import os
from langchain_tavily import TavilySearch

@tool(parse_docstring=True)
def search_recipe_from_10000recipe(menu_name: str) -> str:
    """만개의레시피 사이트에서 특정 메뉴의 레시피 정보, 재료, 조리 순서를 검색합니다.

    Args:
        menu_name: 만들고자 하는 메뉴 이름 (예: 김치찌개, 토마토 파스타)

    Returns:
        만개의레시피 검색 결과 (필수 재료 및 단계별 조리 순서 포함)
    """
    try:
        search = TavilySearch(max_results=5, include_domains=["10000recipe.com"])
        result = search.invoke({"query": f"{menu_name} 레시피 만드는 법"})
        return (
            f"[만개의레시피 검색 결과: {menu_name}]\n"
            f"{result}\n"
            f"※ 위 내용을 바탕으로 필수 재료와 단계별 조리 순서 텍스트를 구성해 주세요."
        )
    except Exception as e:
        return f"레시피 검색 실패: {str(e)}"

@tool(parse_docstring=True)
def search_nearby_restaurants(address: str, menu_name: str) -> str:
    """사용자의 주소 근처에서 특정 메뉴를 파는 맛집을 4~5곳 이상 검색하고 평점과 평가를 요약합니다.

    Args:
        address: 사용자의 현재 주소 또는 동 이름 (예: 수원시 영통구 망포동, 서울 역삼동)
        menu_name: 먹고 싶은 메뉴 이름 (예: 토마토 파스타, 치킨, 초밥)

    Returns:
        근처 맛집 목록 (4~5곳 이상), 간략한 소개, 평점 및 평가 요약
    """
    try:
        search = TavilySearch(max_results=6, topic="general")
        result = search.invoke({"query": f"{address} {menu_name} 맛집 추천 리스트 평점 후기 인기순"})
        return (
            f"[근처 맛집 검색 결과 (다수 추천)]\n"
            f"- 위치: {address} / 메뉴: {menu_name}\n"
            f"{result}\n"
            f"※ 위 검색 결과에서 4~5곳 이상의 다양한 맛집을 추출하여 특징과 평점을 요약해 주세요."
        )
    except Exception as e:
        return f"맛집 검색 실패: {str(e)}"

print("✓ 요리 추천 도메인 도구(만개의레시피, 근처 맛집 검색) 로드 완료")

✓ 요리 추천 도메인 도구(만개의레시피, 근처 맛집 검색) 로드 완료


## 5. 도구 정보 확인

생성된 도구의 메타데이터를 확인합니다.

In [3]:
print("=" * 80)
print("도구 1 정보: 만개의레시피 검색")
print("=" * 80)
print(f"도구 이름: {search_recipe_from_10000recipe.name}")
print(f"도구 설명: {search_recipe_from_10000recipe.description}")
print(f"\n입력 스키마:")
print(search_recipe_from_10000recipe.args_schema.model_json_schema())

print("\n" + "=" * 80)
print("도구 2 정보: 근처 맛집 검색")
print("=" * 80)
print(f"도구 이름: {search_nearby_restaurants.name}")
print(f"도구 설명: {search_nearby_restaurants.description}")
print(f"\n입력 스키마:")
print(search_nearby_restaurants.args_schema.model_json_schema())

도구 1 정보: 만개의레시피 검색
도구 이름: search_recipe_from_10000recipe
도구 설명: 만개의레시피 사이트에서 특정 메뉴의 레시피 정보, 재료, 조리 순서를 검색합니다.

입력 스키마:
{'description': '만개의레시피 사이트에서 특정 메뉴의 레시피 정보, 재료, 조리 순서를 검색합니다.', 'properties': {'menu_name': {'description': '만들고자 하는 메뉴 이름 (예: 김치찌개, 토마토 파스타)', 'title': 'Menu Name', 'type': 'string'}}, 'required': ['menu_name'], 'title': 'search_recipe_from_10000recipe', 'type': 'object'}

도구 2 정보: 근처 맛집 검색
도구 이름: search_nearby_restaurants
도구 설명: 사용자의 주소 근처에서 특정 메뉴를 파는 맛집을 4~5곳 이상 검색하고 평점과 평가를 요약합니다.

입력 스키마:
{'description': '사용자의 주소 근처에서 특정 메뉴를 파는 맛집을 4~5곳 이상 검색하고 평점과 평가를 요약합니다.', 'properties': {'address': {'description': '사용자의 현재 주소 또는 동 이름 (예: 수원시 영통구 망포동, 서울 역삼동)', 'title': 'Address', 'type': 'string'}, 'menu_name': {'description': '먹고 싶은 메뉴 이름 (예: 토마토 파스타, 치킨, 초밥)', 'title': 'Menu Name', 'type': 'string'}}, 'required': ['address', 'menu_name'], 'title': 'search_nearby_restaurants', 'type': 'object'}


## 6. 도구 단독 실행 테스트

**TODO: 다양한 입력값으로 도구를 테스트하세요**

테스트 케이스를 최소 3개 이상 작성하세요:
1. 정상 케이스
2. 엣지 케이스 (경계값)
3. 에러 케이스 (잘못된 입력)

In [4]:
print("=" * 80)
print("도구 단독 실행 테스트")
print("=" * 80)

print("\n[테스트 1] 만개의레시피 검색 테스트 (정상 케이스)")
result1 = search_recipe_from_10000recipe.invoke({"menu_name": "토마토 파스타"})
print(result1)
print("-" * 50)

print("\n[테스트 2] 근처 맛집 검색 테스트 (정상 케이스 - 서울 서초구")
result2 = search_nearby_restaurants.invoke({"address": "서울 서초구", "menu_name": "스파게티"})
print(result2)
print("-" * 50)

print("\n[테스트 3] 근처 맛집 검색 테스트 (다른 지역 - 역삼동)")
result3 = search_nearby_restaurants.invoke({"address": "서울 역삼동", "menu_name": "토마토 파스타"})
print(result3)

도구 단독 실행 테스트

[테스트 1] 만개의레시피 검색 테스트 (정상 케이스)
[만개의레시피 검색 결과: 토마토 파스타]
{'query': '토마토 파스타 레시피 만드는 법', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.10000recipe.com/recipe/7039684', 'title': '토마토파스타 만들기~ 시판소스 더 건강하게 만든 파스타~', 'content': '.\n\n토마토파스타\n\n휘리릭 만드는 주말메뉴 토마토파스타\n\n토마토파스타만들기 레시피\n\n파스타 180g, 소금1T\n\n양파1/4개, 다진마늘1T\n\n토마토소스 1.5~2컵, 완숙토마토1개,\n\n냉동새우 6개\n\n베트남고추 3개\n\n파슬리,\n\n2인분 양입니다.\n\n> 파스타 만들기\n\n면 길이보다 큰 팬에\n\n물 끓여 소금 넣고\n\n파스타 면 넣고\n\n봉지에 써있는 시간만큼 삶아요~\n\n전 스파게티니라 7분\n\n냉동 새우도 함께 넣어 해동시킵니다.\n\n팬에 올리브유 두르고\n\n다진양파,다진마늘 볶아요~\n\n월남고추 대충 부셔넣고\n\n매운거 못 먹는 사람 건지기 쉽게\n\n새우도 건져 넣고\n\n냉동실에 얼린 완숙토마토\n\n녹이지 않고 대충 썰어서 넣어요~\n\n녹으며 다 풀어집니다.\n\n시판소스 좀더 건강하게 먹는방법은\n\n생토마토를 넣는거랍니다.\n\n토마토소스 넣고\n\n제가 이태리 요리 배울때는\n\n1인분에 소스 반컵 정도라고 배웠지만\n\n우리는 소스 많은 걸 좋아하니 듬뿍~\n\n소스끓인팬에\n\n삶은 파스타면 건져넣고\n\n너무 뻑벅하지않게 [...] 돼지고기 1토막\n\n    \n\n    약 100g\n\n    양파 1개 (중간사이즈)\n\n    \n\n    약 170g\n\n    당근 1개 (중간사이즈)\n\n    \n\n    약 150g\n\n    무 1토막 (약 2cm)\n\n    \

## 7. 여러 도구 통합 테스트

팀에서 만든 여러 도구를 함께 테스트합니다.

**TODO: 생성한 모든 도구를 리스트로 정리하세요**

In [5]:
# 생성한 모든 도구를 리스트로 통합
RECIPE_TOOLS = [
    search_recipe_from_10000recipe,
    search_nearby_restaurants,
]

print(f"총 {len(RECIPE_TOOLS)}개의 도구가 준비되었습니다.\n")

for i, tool in enumerate(RECIPE_TOOLS, 1):
    print(f"{i}. {tool.name}")
    print(f"   설명: {tool.description}")
    print()

총 2개의 도구가 준비되었습니다.

1. search_recipe_from_10000recipe
   설명: 만개의레시피 사이트에서 특정 메뉴의 레시피 정보, 재료, 조리 순서를 검색합니다.

2. search_nearby_restaurants
   설명: 사용자의 주소 근처에서 특정 메뉴를 파는 맛집을 4~5곳 이상 검색하고 평점과 평가를 요약합니다.



## 프로젝트 체크리스트

**완료한 항목을 확인하세요:**

- [ ] 팀 도메인 선정 및 필요한 도구 정의 완료
- [ ] AI 프롬프트를 사용하여 도구 코드 생성 완료
- [ ] 최소 3개 이상의 도구 생성 완료
- [ ] 각 도구별 단독 실행 테스트 완료
- [ ] 정상/엣지/에러 케이스 테스트 완료
- [ ] 도구 메타데이터 확인 완료

---

## 다음 단계

생성한 도구를 domain-agent에 통합하세요:

1. `../src/domain-agent/tools.py` 파일 열기
2. TODO 주석을 참고하여 생성한 도구 코드 추가
3. `../src/domain-agent/agent.py` 파일 열기
4. TODO 주석을 참고하여 시스템 프롬프트와 도구 리스트 수정
5. LangGraph Studio로 테스트

---

## 참고 자료

- [LangChain Tools 문서](https://python.langchain.com/docs/modules/agents/tools/)
- [LangChain Custom Tools](https://python.langchain.com/docs/modules/agents/tools/custom_tools/)
- [@tool 데코레이터](https://python.langchain.com/docs/modules/agents/tools/custom_tools/#tool-decorator)